In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import pandas as pd

In [2]:
df = pd.read_csv('dataset/pond_iot_2023.csv')
df.head()

,id,created_date,water_pH,TDS,water_temp
0,181775,1/26/2023 11:24,7.60,250,24.00
1,181777,1/26/2023 11:33,7.60,247,24.06
2,181778,1/26/2023 12:02,7.60,249,24.19
3,181780,1/26/2023 12:05,6.92,247,24.19
4,181782,1/26/2023 12:10,7.30,245,24.25


In [3]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
import os

def preprocess_and_resample(file_path="dataset/pond_iot_2023.csv", freq="1h"):
    """
    Loads the dataset, parses timestamps, resamples to a fixed frequency (freq),
    interpolates gaps, and returns a clean DataFrame.
    """
    print(f"Loading {file_path}...")
    df = pd.read_csv(file_path)
    
    # Parse date
    df['created_date'] = pd.to_datetime(df['created_date'])
    df = df.sort_values('created_date')
    
    # Drop id column
    df = df.drop(columns=['id'])
    
    # Set created_date as index
    df = df.set_index('created_date')
    
    # Resample using mean
    print(f"Resampling to frequency '{freq}'...")
    df_resampled = df.resample(freq).mean()
    
    # Check for missing values after resampling (due to offline gaps)
    missing_before = df_resampled.isnull().sum()
    print(f"Missing values before interpolation:\n{missing_before}")
    
    # Interpolate missing values linearly
    df_resampled = df_resampled.interpolate(method='linear')
    
    # If there are still any NaNs at the beginning, fill them backward
    df_resampled = df_resampled.bfill()
    
    missing_after = df_resampled.isnull().sum()
    print(f"Missing values after interpolation:\n{missing_after}")
    print(f"Resampled dataset size: {df_resampled.shape}")
    
    return df_resampled

In [7]:
df = preprocess_and_resample()
df.head()

Loading dataset/pond_iot_2023.csv...
Resampling to frequency '1h'...
Missing values before interpolation:
water_pH      837
TDS           837
water_temp    837
dtype: int64
Missing values after interpolation:
water_pH      0
TDS           0
water_temp    0
dtype: int64
Resampled dataset size: (1327, 3)


,water_pH,TDS,water_temp
created_date,,,
2023-01-26 11:00:00,7.600000,248.500000,24.030000
2023-01-26 12:00:00,7.383125,248.062500,24.324375
2023-01-26 13:00:00,7.442857,250.857143,24.598571
2023-01-26 14:00:00,7.463229,248.645833,25.056979
2023-01-26 15:00:00,7.406786,250.035714,25.202054


In [8]:
def prepare_lstm_data(df, lookback=24, forecast_horizon=1, train_ratio=0.8):
    """
    Scales the features and splits them into training/testing sequences.
    lookback: Number of past time steps to look back (input sequence length)
    forecast_horizon: Number of future time steps to predict
    """
    features = ['water_pH', 'TDS', 'water_temp']
    data = df[features].values
    
    # Scale data to [0, 1] range (crucial for LSTM)
    scaler = MinMaxScaler()
    scaled_data = scaler.fit_transform(data)
    
    X, y = [], []
    for i in range(len(scaled_data) - lookback - forecast_horizon + 1):
        X.append(scaled_data[i : i + lookback])
        # We predict the 3 target variables at 'i + lookback' up to 'i + lookback + forecast_horizon'
        y.append(scaled_data[i + lookback : i + lookback + forecast_horizon])
        
    X = np.array(X)
    y = np.array(y)
    
    # Reshape y if forecast_horizon is 1 to (N, 3) instead of (N, 1, 3)
    if forecast_horizon == 1:
        y = y.squeeze(axis=1)
        
    # Split into train and test sets chronologically
    train_size = int(len(X) * train_ratio)
    
    X_train, X_test = X[:train_size], X[train_size:]
    y_train, y_test = y[:train_size], y[train_size:]
    
    print(f"X_train shape: {X_train.shape}, y_train shape: {y_train.shape}")
    print(f"X_test shape: {X_test.shape}, y_test shape: {y_test.shape}")
    
    return X_train, y_train, X_test, y_test, scaler

In [27]:
X_train, y_train, X_test, y_test, scaler = prepare_lstm_data(df)

X_train shape: (1042, 24, 3), y_train shape: (1042, 3)
X_test shape: (261, 24, 3), y_test shape: (261, 3)


In [30]:
import torch
import torch.nn as nn

class WaterQualityLSTM(nn.Module):
    def __init__(self, input_size=3, hidden_size=64, num_layers=2, output_size=3, dropout=0.2):
        """
        LSTM Model for Water Quality time series forecasting.
        - input_size: number of features (3: water_pH, TDS, water_temp)
        - hidden_size: number of hidden units in LSTM cell
        - num_layers: number of stacked LSTM layers
        - output_size: number of features to predict (3)
        - dropout: dropout probability to prevent overfitting
        """
        super(WaterQualityLSTM, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        
        # LSTM Layer
        # batch_first=True means the input/output tensors are of shape (batch, seq, feature)
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0
        )
        
        # Fully connected layers to output predictions
        self.fc = nn.Sequential(
            nn.Linear(hidden_size, 32),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(32, output_size)
        )
        
    def forward(self, x):
        # x shape: (batch_size, seq_len, input_size)
        
        # Initialize hidden state (h0) and cell state (c0) with zeros
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(x.device)
        c0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(x.device)
        
        # Pass through LSTM
        # out shape: (batch_size, seq_len, hidden_size)
        out, _ = self.lstm(x, (h0, c0))
        
        # Get the output from the last time step
        # out shape: (batch_size, hidden_size)
        out_last = out[:, -1, :]
        
        # Pass through the regression head
        # prediction shape: (batch_size, output_size)
        prediction = self.fc(out_last)
        
        return prediction

In [35]:
from torch.utils.data import Dataset, DataLoader

class MyDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)
        
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]
    
train_loader = DataLoader(MyDataset(X_train, y_train), batch_size=32, shuffle=True)
test_loader = DataLoader(MyDataset(X_test, y_test), batch_size=32, shuffle=False)

In [36]:
epochs = 100
lr=0.001
patience = 15
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = WaterQualityLSTM()
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    
train_losses = []
test_losses = []

best_loss = float('inf')
patience_counter = 0
best_model_weights = None

print("\n--- Training LSTM Model ---")
for epoch in range(1, epochs + 1):
    model.train()
    epoch_train_losses = []
    
    for batch_X, batch_y in train_loader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
        
        # Forward pass
        predictions = model(batch_X)
        loss = criterion(predictions, batch_y)
        
        # Backward pass and optimize
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        epoch_train_losses.append(loss.item())
        
    # Evaluate on test set
    model.eval()
    epoch_test_losses = []
    
    with torch.no_grad():
        for batch_X, batch_y in test_loader:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)
            predictions = model(batch_X)
            test_loss = criterion(predictions, batch_y)
            epoch_test_losses.append(test_loss.item())
            
    mean_train_loss = np.mean(epoch_train_losses)
    mean_test_loss = np.mean(epoch_test_losses)
    
    train_losses.append(mean_train_loss)
    test_losses.append(mean_test_loss)
    
    if epoch % 5 == 0 or epoch == 1:
        print(f"Epoch {epoch}/{epochs} | Train MSE Loss: {mean_train_loss:.6f} | Test MSE Loss: {mean_test_loss:.6f}")
        
    # Early stopping check
    if mean_test_loss < best_loss:
        best_loss = mean_test_loss
        patience_counter = 0
        best_model_weights = model.state_dict().copy()
        # Save the best model
        torch.save(best_model_weights, "artifacts/lstm_water_quality.pth")
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print(f"\nEarly stopping triggered at epoch {epoch}! Best Test MSE Loss: {best_loss:.6f}")
            break
            
# Load best weights back to the model
if best_model_weights is not None:
    model.load_state_dict(best_model_weights)
    print("Loaded best weights from training phase.")


--- Training LSTM Model ---
Epoch 1/100 | Train MSE Loss: 0.172855 | Test MSE Loss: 0.101323
Epoch 5/100 | Train MSE Loss: 0.030348 | Test MSE Loss: 0.025330
Epoch 10/100 | Train MSE Loss: 0.017410 | Test MSE Loss: 0.018648
Epoch 15/100 | Train MSE Loss: 0.012621 | Test MSE Loss: 0.008800
Epoch 20/100 | Train MSE Loss: 0.011491 | Test MSE Loss: 0.008161
Epoch 25/100 | Train MSE Loss: 0.009540 | Test MSE Loss: 0.007356
Epoch 30/100 | Train MSE Loss: 0.008841 | Test MSE Loss: 0.008199
Epoch 35/100 | Train MSE Loss: 0.008177 | Test MSE Loss: 0.008009

Early stopping triggered at epoch 39! Best Test MSE Loss: 0.006430
Loaded best weights from training phase.
